# SingBERT Scam / Non-Scam Classifier

This notebook is written to be run top-to-bottom. It does four things:

1. combines the six raw CSV files in `datasets/` into one canonical dataset;
2. cleans exact duplicates and blank rows before modeling;
3. fine-tunes a Hugging Face SingBERT checkpoint for multi-class text classification using the `label` column;
4. saves the merged CSV, fine-tuned model, tokenizer, label mapping, and basic train / validation / test metrics.

The default classifier is multi-class because your raw data currently contains four labels: `authority_scam`, `job_scam`, `love_scam`, and `not_scam`.


## SingBERT Notes

Why this notebook uses `zanelim/singbert`:

- SingBERT is not a brand-new architecture. Its published config keeps the standard BERT-Base encoder shape: `12` transformer layers, `12` attention heads, hidden size `768`, max position embeddings `512`, and vocab size `30522`.
- The model card says it starts from `bert-base-uncased`, then continues pre-training on colloquial Singlish and Manglish text collected from `r/singapore`, `r/malaysia`, and the HardwareZone forum.
- The author also states that the top `1000` custom local tokens that do not overlap with the original BERT vocabulary were inserted into unused vocabulary slots.
- Continued pre-training used `train_batch_size=512`, `max_seq_length=128`, `num_train_steps=300000`, `num_warmup_steps=5000`, and `learning_rate=2e-5` on a TPU v3-8.
- For classification, the workflow is standard BERT fine-tuning: tokenize the sentence with the model tokenizer, run it through the bidirectional transformer encoder, and attach a sequence-classification head on top of the pooled sequence representation.

Primary sources:

- SingBERT model card: https://huggingface.co/zanelim/singbert
- SingBERT config: https://huggingface.co/zanelim/singbert/raw/main/config.json
- Original BERT repository and model notes: https://github.com/google-research/bert
- Transformers installation notes: https://huggingface.co/docs/transformers/installation


In [ ]:
# Only install packages that Colab does not ship with.
# Do NOT reinstall torch / torchvision — Colab's pre-installed versions are
# already built against the same CUDA toolkit.  Upgrading one without the
# other causes the CUDA-major-version mismatch error.
%pip install -q transformers datasets accelerate

from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "datasets"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
MODEL_OUTPUT_DIR = ARTIFACT_DIR / "singbert_classifier"
METRICS_OUTPUT_DIR = ARTIFACT_DIR / "metrics"

RAW_COMBINED_PATH = DATA_DIR / "combined_dataset_raw.csv"
COMBINED_INPUT_PATH = DATA_DIR / "combined_dataset.csv"

EXPECTED_COLUMNS = ["text", "label", "persona", "source_type", "scenario"]
TEXT_COLUMN = "text"
LABEL_COLUMN = "label"

EXPECTED_SOURCE_FILE_COUNT = 6
MODEL_CHECKPOINT = "zanelim/singbert"

RANDOM_SEED = 42
TEST_SIZE = 0.15
VALIDATION_SIZE = 0.15

# SingBERT's published continued-pretraining used max_seq_length=128,
# so the default classifier setup keeps the same sequence length.
MAX_LENGTH = 128
NUM_EPOCHS = 4
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
WEIGHT_DECAY = 0.01

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(RANDOM_SEED)
pd.set_option("display.max_colwidth", 160)

In [ ]:
csv_paths = sorted(DATA_DIR.glob("*.csv"))
source_paths = [
    path
    for path in csv_paths
    if path.name not in {RAW_COMBINED_PATH.name, COMBINED_INPUT_PATH.name}
]

if len(source_paths) != EXPECTED_SOURCE_FILE_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_SOURCE_FILE_COUNT} source CSV files, found {len(source_paths)}: "
        f"{[path.name for path in source_paths]}"
    )

frames = []
for path in source_paths:
    frame = pd.read_csv(path)

    missing_columns = [col for col in EXPECTED_COLUMNS if col not in frame.columns]
    extra_columns = [col for col in frame.columns if col not in EXPECTED_COLUMNS]

    if missing_columns:
        raise ValueError(f"{path.name} is missing required columns: {missing_columns}")

    if extra_columns:
        print(f"Warning: {path.name} has extra columns that will be dropped: {extra_columns}")

    frame = frame[EXPECTED_COLUMNS].copy()
    frame["source_file"] = path.name
    frames.append(frame)

combined_raw_df = pd.concat(frames, ignore_index=True)
combined_raw_df.to_csv(RAW_COMBINED_PATH, index=False)

combined_df = combined_raw_df.copy()
combined_df[TEXT_COLUMN] = combined_df[TEXT_COLUMN].astype(str).str.strip()
combined_df[LABEL_COLUMN] = combined_df[LABEL_COLUMN].astype(str).str.strip()

combined_df = combined_df.loc[
    combined_df[TEXT_COLUMN].ne("") & combined_df[LABEL_COLUMN].ne("")
].drop_duplicates(subset=EXPECTED_COLUMNS).reset_index(drop=True)

combined_df.to_csv(COMBINED_INPUT_PATH, index=False)

merge_summary = pd.DataFrame(
    {
        "file": [path.name for path in source_paths],
        "rows": [len(frame) for frame in frames],
    }
)

display(merge_summary)
print(f"Raw merged rows: {len(combined_raw_df):,}")
print(f"Clean merged rows: {len(combined_df):,}")
print(f"Saved raw merge to: {RAW_COMBINED_PATH}")
print(f"Saved clean merge to: {COMBINED_INPUT_PATH}")

display(combined_df.head())
display(combined_df[LABEL_COLUMN].value_counts().rename_axis("label").to_frame("count"))


In [ ]:
model_df = pd.read_csv(COMBINED_INPUT_PATH)

required_columns = {TEXT_COLUMN, LABEL_COLUMN}
missing_required = required_columns - set(model_df.columns)
if missing_required:
    raise ValueError(f"The combined CSV is missing required columns: {sorted(missing_required)}")

for optional_column in ["persona", "source_type", "scenario", "source_file"]:
    if optional_column not in model_df.columns:
        model_df[optional_column] = "unknown"

model_df[TEXT_COLUMN] = model_df[TEXT_COLUMN].astype(str).str.strip()
model_df[LABEL_COLUMN] = model_df[LABEL_COLUMN].astype(str).str.strip()

dedupe_columns = [col for col in EXPECTED_COLUMNS if col in model_df.columns]
model_df = model_df.loc[
    model_df[TEXT_COLUMN].ne("") & model_df[LABEL_COLUMN].ne("")
].drop_duplicates(subset=dedupe_columns).reset_index(drop=True)

label_names = sorted(model_df[LABEL_COLUMN].unique().tolist())
label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for label, idx in label2id.items()}
model_df["labels"] = model_df[LABEL_COLUMN].map(label2id)

train_val_df, test_df = train_test_split(
    model_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=model_df["labels"],
)

validation_relative_size = VALIDATION_SIZE / (1 - TEST_SIZE)
train_df, validation_df = train_test_split(
    train_val_df,
    test_size=validation_relative_size,
    random_state=RANDOM_SEED,
    stratify=train_val_df["labels"],
)

split_frames = {"train": train_df, "validation": validation_df, "test": test_df}
for split_name, split_df in split_frames.items():
    print(f"{split_name:>10}: {len(split_df):4d} rows")
    print(split_df[LABEL_COLUMN].value_counts().sort_index().to_string())
    print()

dataset_dict = DatasetDict(
    {
        split_name: Dataset.from_pandas(split_df.reset_index(drop=True), preserve_index=False)
        for split_name, split_df in split_frames.items()
    }
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)

def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COLUMN],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_datasets = dataset_dict.map(tokenize_batch, batched=True)

tensor_columns = ["input_ids", "attention_mask", "labels"]
if "token_type_ids" in tokenized_datasets["train"].column_names:
    tensor_columns.append("token_type_ids")

tokenized_datasets.set_format(type="torch", columns=tensor_columns)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

label_mapping_path = METRICS_OUTPUT_DIR / "label_mapping.json"
with open(label_mapping_path, "w", encoding="utf-8") as fp:
    json.dump(
        {
            "label2id": label2id,
            "id2label": {str(k): v for k, v in id2label.items()},
        },
        fp,
        indent=2,
    )

print(f"Saved label mapping to: {label_mapping_path}")


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
    }

training_args_kwargs = dict(
    output_dir=str(MODEL_OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    save_total_limit=2,
    seed=RANDOM_SEED,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

if "eval_strategy" in TrainingArguments.__dataclass_fields__:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_args_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()

trainer.save_model(str(MODEL_OUTPUT_DIR))
tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR))

def to_python_scalars(metrics_dict):
    cleaned = {}
    for key, value in metrics_dict.items():
        if isinstance(value, (np.floating, np.integer)):
            cleaned[key] = value.item()
        else:
            cleaned[key] = value
    return cleaned

def evaluate_split(split_name, dataset_split):
    prediction_output = trainer.predict(dataset_split, metric_key_prefix=split_name)
    predicted_ids = prediction_output.predictions.argmax(axis=-1)
    true_ids = prediction_output.label_ids

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        true_ids, predicted_ids, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        true_ids, predicted_ids, average="weighted", zero_division=0
    )

    split_metrics = to_python_scalars(
        {
            "loss": prediction_output.metrics.get(f"{split_name}_loss"),
            "accuracy": accuracy_score(true_ids, predicted_ids),
            "precision_macro": precision_macro,
            "recall_macro": recall_macro,
            "f1_macro": f1_macro,
            "precision_weighted": precision_weighted,
            "recall_weighted": recall_weighted,
            "f1_weighted": f1_weighted,
        }
    )

    report = classification_report(
        true_ids,
        predicted_ids,
        labels=list(range(len(label2id))),
        target_names=[id2label[idx] for idx in range(len(label2id))],
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report).transpose()
    report_path = METRICS_OUTPUT_DIR / f"{split_name}_classification_report.csv"
    report_df.to_csv(report_path)

    return split_metrics, report_df

validation_metrics = to_python_scalars(
    trainer.evaluate(tokenized_datasets["validation"], metric_key_prefix="validation")
)
train_metrics, train_report_df = evaluate_split("train", tokenized_datasets["train"])
test_metrics, test_report_df = evaluate_split("test", tokenized_datasets["test"])

summary_metrics = {
    "model_checkpoint": MODEL_CHECKPOINT,
    "combined_input_path": str(COMBINED_INPUT_PATH),
    "output_model_dir": str(MODEL_OUTPUT_DIR),
    "split_sizes": {split_name: len(split_df) for split_name, split_df in split_frames.items()},
    "best_model_checkpoint": trainer.state.best_model_checkpoint,
    "train_runtime": to_python_scalars(train_result.metrics),
    "validation": validation_metrics,
    "train": train_metrics,
    "test": test_metrics,
}

summary_metrics_path = METRICS_OUTPUT_DIR / "summary_metrics.json"
with open(summary_metrics_path, "w", encoding="utf-8") as fp:
    json.dump(summary_metrics, fp, indent=2)

trainer_log_path = METRICS_OUTPUT_DIR / "trainer_log_history.json"
with open(trainer_log_path, "w", encoding="utf-8") as fp:
    json.dump([to_python_scalars(item) for item in trainer.state.log_history], fp, indent=2)

metrics_overview = pd.DataFrame([train_metrics, test_metrics], index=["train", "test"])
display(metrics_overview)
display(test_report_df)

print(f"Saved fine-tuned model to: {MODEL_OUTPUT_DIR}")
print(f"Saved metric summary to: {summary_metrics_path}")
print(f"Saved trainer history to: {trainer_log_path}")
print(f"Saved per-class reports to: {METRICS_OUTPUT_DIR}")


## Download the trained model

Zip only the files needed for inference (skips checkpoint folders and training args) so you can download a single file.

In [ ]:
import shutil

# Only the files actually written by trainer.save_model + tokenizer.save_pretrained
# for this fast-tokenizer setup.  Checkpoints and training_args.bin are dropped.
INFERENCE_FILES = [
    "config.json",
    "model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
]

EXPORT_DIR = ARTIFACT_DIR / "singbert_classifier_export"
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True)

for filename in INFERENCE_FILES:
    shutil.copy2(MODEL_OUTPUT_DIR / filename, EXPORT_DIR / filename)

# Bundle the label mapping alongside the model so consumers know what each id means.
shutil.copy2(label_mapping_path, EXPORT_DIR / "label_mapping.json")

archive_base = ARTIFACT_DIR / "singbert_classifier_export"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", EXPORT_DIR))

print(f"Zipped model to: {archive_path}")
print(f"Archive size: {archive_path.stat().st_size / (1024 * 1024):.1f} MB")

# On Google Colab, uncomment the next two lines to trigger a browser download.
# from google.colab import files
# files.download(str(archive_path))

## Inference

Load the fine-tuned model from disk and classify new messages. This reads the saved model directory so it works both right after training and later, after restarting the runtime (as long as `artifacts/singbert_classifier/` exists).

In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# --- Load model, tokenizer, and label mapping from the saved artifacts ---
_model_dir = Path("artifacts/singbert_classifier")
_label_map_path = Path("artifacts/metrics/label_mapping.json")

with open(_label_map_path, encoding="utf-8") as f:
    _label_map = json.load(f)
_id2label = {int(k): v for k, v in _label_map["id2label"].items()}

_tokenizer = AutoTokenizer.from_pretrained(_model_dir)
_model = AutoModelForSequenceClassification.from_pretrained(_model_dir)
_model.eval()

_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_model.to(_device)


def classify(text: str) -> dict:
    """Return the predicted label and per-class probabilities for a single message."""
    inputs = _tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(_device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = _model(**inputs).logits

    probs = torch.softmax(logits, dim=-1).squeeze().cpu().tolist()
    predicted_id = int(torch.argmax(logits, dim=-1))

    return {
        "text": text,
        "prediction": _id2label[predicted_id],
        "probabilities": {_id2label[i]: round(p, 4) for i, p in enumerate(probs)},
    }


# --- Try it out ---
sample_messages = [
    "Eh bro, want to go makan later at the hawker centre?",
    "Congratulations! You have been selected for a $5000 reward. Click here to claim now.",
    "Hey darling, I miss you so much. Can you send me money for flight ticket to come see you?",
    "Bro later meet at MRT ah, 7pm can?",
]

for msg in sample_messages:
    result = classify(msg)
    print(f"Text:       {result['text']}")
    print(f"Prediction: {result['prediction']}")
    print(f"Probs:      {result['probabilities']}")
    print()